# Hybrid Search (Dense Semantic + Sparse Lexical BM25)

Dense vector embeddings excel at conceptual semantic search but can miss exact keyword matches, code identifiers, and acronyms. Hybrid search unites dense FAISS retrieval and sparse BM25 lexical scoring through Reciprocal Rank Fusion (RRF) using LangChain's EnsembleRetriever.

## Workflow Architecture

<div align="center">
  <img src="workflow_hybrid_search.png" alt="Hybrid Search (Dense Semantic + Sparse Lexical BM25) Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["User Query"]
        Q(["1. User Query"]):::startNode
    end
    subgraph DualRetrieval["Dual Retrieval Channels"]
        Dense["2a. Dense Semantic Retrieval<br/><b>FAISS + MiniLM</b><br/>(Captures Conceptual Meaning)"]:::denseNode
        Sparse["2b. Sparse Lexical Retrieval<br/><b>BM25 Okapi</b><br/>(Captures Exact Keywords and Acronyms)"]:::sparseNode
    end
    subgraph RankFusion["Reciprocal Rank Fusion (RRF)"]
        RRF["3. Reciprocal Rank Fusion Algorithm<br/><b>Score = 1 / (60 + Rank)</b><br/>(Harmonizes Dense and Sparse Rankings)"]:::rrfNode
        FusedDocs["4. Unified Top-Ranked Context Chunks"]:::fusedNode
    end
    subgraph Out["Final Output"]
        LLM["5. Groq LLM Synthesis"]:::llmNode
        Ans(["6. Robust Hybrid Answer"]):::endNode
    end
    Q --> Dense
    Q --> Sparse
    Dense --> RRF
    Sparse --> RRF
    RRF --> FusedDocs
    FusedDocs --> LLM
    Q --> LLM
    LLM --> Ans
    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef denseNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef sparseNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef rrfNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef fusedNode fill:#E8EAF6,stroke:#3F51B5,stroke-width:2px,color:#1A237E;
    classDef llmNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef endNode fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
```
</details>

### Key Retrieval Principles
- **Best of Both Worlds**: Captures semantic nuances via MiniLM embeddings while ensuring exact term matches via BM25.
- **Reciprocal Rank Fusion**: Standardized rank scoring (1 / (60 + rank)) merges diverse result lists without scale bias.
- **Enterprise Resilience**: Particularly effective for queries containing rare product names, codes, or technical jargon.


In [29]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
try:
    from langchain.retrievers import EnsembleRetriever
except (ImportError, ModuleNotFoundError):
    from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_core.documents import Document

docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

In [30]:
from langchain_community import vectorstores
# 1. Dense Semantic Retriever: HuggingFace Embeddings + FAISS
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embeddings)
dense_retriever = dense_vectorstore.as_retriever(search_kwargs={"k": 2})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9136.56it/s]


In [31]:
# 2. Sparse Keyword Retriever: BM25
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 2

In [32]:
# 3. Hybrid Retriever: Combine Sparse (BM25) and Dense (FAISS) using Reciprocal Rank Fusion (RRF)
ensemble_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
)

In [33]:
# 4. Query the Hybrid Retriever
query = "How can I build an application using LLMs?"
results = ensemble_retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"[{i}] {doc.page_content}")

[1] LangChain helps build LLM applications.
[2] Langchain can be used to develop agentic ai application.


In [34]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain.chat_models import init_chat_model
# Create a RAG pipeline

# Use the most stable, globally available model
llm = init_chat_model(model="groq:openai/gpt-oss-120b")

prompt = PromptTemplate.from_template(""" 
Answer the question based on the context below.
Context: {context}
Question: {input}
""")
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=ensemble_retriever, combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001894347F530>, search_kwargs={'k': 2}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000018944D343E0>, k=2)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template=' \nAnswer the question based on the context below.\nContext: {context}\nQuestion: {input}\n')
            | ChatGroq(metadat

In [35]:
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)
print("■ Answer:\n", response["answer"])

■ Answer:
 Below is a practical, step‑by‑step guide to building an application that uses a large language model (LLM) – for example, OpenAI’s GPT‑4 – with **LangChain**.  
LangChain is a library that abstracts away a lot of the plumbing (prompt management, LLM calls, memory, tool integration, routing, etc.) so you can focus on the product logic.

---

## 1. Define the App’s Goal & Architecture

| Question | What to decide |
|----------|----------------|
| **What does the app do?** | e.g., “Answer user questions about a product catalog”, “Summarize uploaded PDFs”, “Plan a travel itinerary”, “Chat‑assistant for a SaaS dashboard”. |
| **Is it a simple Q&A or an *agentic* workflow?** | Simple: one LLM call → answer.<br>Agentic: LLM decides which tool to call (search, database, calculator, API) and may iterate. |
| **What data sources are needed?** | Static text, vector store, external APIs, databases, file uploads, etc. |
| **User interface?** | Web (React/Next.js), mobile, CLI, Slack/Disc